# 动量方法与自适应学习率

本notebook介绍优化算法的两大改进方向:动量方法和自适应学习率。

## 学习目标

- 理解动量法如何加速收敛
- 掌握AdaGrad的自适应学习率机制
- 学习RMSProp对AdaGrad的改进
- 理解Adam算法的设计思想
- 了解Adadelta的无学习率特性

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
import math

torch.manual_seed(42)
np.random.seed(42)

## 1. 动量法(Momentum)

### 1.1 SGD的问题

标准SGD的更新:
$$
\mathbf{w}_t = \mathbf{w}_{t-1} - \eta \mathbf{g}_t
$$

**问题**:
1. **振荡**: 在陡峭方向上来回振荡
2. **缓慢**: 在平坦方向上进展缓慢
3. **噪声**: 梯度噪声导致路径曲折

### 1.2 动量法的思想

**物理类比**: 一个球从山坡滚下
- 积累动量(速度)
- 惯性使其能冲过小坑
- 减少振荡

**数学形式**:
$$
\begin{aligned}
\mathbf{v}_t &= \beta \mathbf{v}_{t-1} + \mathbf{g}_t \\
\mathbf{w}_t &= \mathbf{w}_{t-1} - \eta \mathbf{v}_t
\end{aligned}
$$

其中:
- $\mathbf{v}_t$: 速度(velocity)
- $\beta$: 动量系数,通常取0.9
- $\mathbf{g}_t$: 当前梯度

### 1.3 指数加权移动平均

展开速度项:
$$
\mathbf{v}_t = \mathbf{g}_t + \beta \mathbf{g}_{t-1} + \beta^2 \mathbf{g}_{t-2} + \cdots = \sum_{\tau=0}^{t-1} \beta^\tau \mathbf{g}_{t-\tau}
$$

**有效时间窗口**: $\frac{1}{1-\beta}$
- $\beta=0.9$: 约10步的平均
- $\beta=0.99$: 约100步的平均

**权重总和**: $\sum_{\tau=0}^\infty \beta^\tau = \frac{1}{1-\beta}$

In [ ]:
# 可视化指数加权移动平均的权重分布
betas = [0.5, 0.9, 0.95, 0.99]
time_steps = torch.arange(40)

plt.figure(figsize=(12, 6))
for beta in betas:
    weights = (1 - beta) * (beta ** time_steps)
    plt.plot(time_steps, weights, linewidth=2, 
             label=f'β={beta} (窗口≈{1/(1-beta):.1f}步)')

plt.xlabel('过去时间步 t')
plt.ylabel('权重')
plt.title('动量法中过去梯度的权重分布')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("观察:")
print("- β越大,记忆越长,平滑效果越强")
print("- β=0.9时,最近10步梯度贡献了大部分权重")
print("- β=0.99时,平均了约100步的梯度")

### 1.4 病态条件问题(Ill-conditioned Problems)

**定义**: 不同方向上曲率差异很大的优化问题

**例子**: 
$$
f(x_1, x_2) = 0.1x_1^2 + 2x_2^2
$$

- $x_1$方向: 曲率小(0.2),梯度变化慢
- $x_2$方向: 曲率大(4.0),梯度变化快

**SGD的问题**: 在陡峭方向振荡,在平坦方向进展慢

**动量法的优势**: 平滑振荡,加速平坦方向

In [ ]:
# 对比SGD和Momentum在病态问题上的表现
def f_ill_conditioned(x1, x2):
    """病态目标函数"""
    return 0.1 * x1**2 + 2 * x2**2

def sgd_2d(lr, num_iters=20, x_init=(3.0, 3.0)):
    """标准SGD"""
    x1, x2 = x_init
    trajectory = [(x1, x2)]
    
    for _ in range(num_iters):
        g1 = 0.2 * x1  # ∂f/∂x1
        g2 = 4.0 * x2  # ∂f/∂x2
        x1 -= lr * g1
        x2 -= lr * g2
        trajectory.append((x1, x2))
    
    return np.array(trajectory)

def momentum_2d(lr, beta, num_iters=20, x_init=(3.0, 3.0)):
    """动量法"""
    x1, x2 = x_init
    v1, v2 = 0.0, 0.0
    trajectory = [(x1, x2)]
    
    for _ in range(num_iters):
        g1 = 0.2 * x1
        g2 = 4.0 * x2
        
        # 更新速度
        v1 = beta * v1 + g1
        v2 = beta * v2 + g2
        
        # 更新参数
        x1 -= lr * v1
        x2 -= lr * v2
        trajectory.append((x1, x2))
    
    return np.array(trajectory)

# 生成等高线
x1_range = np.linspace(-4, 4, 100)
x2_range = np.linspace(-4, 4, 100)
X1, X2 = np.meshgrid(x1_range, x2_range)
Z = 0.1 * X1**2 + 2 * X2**2

# 对比可视化
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# SGD轨迹
traj_sgd = sgd_2d(lr=0.4, num_iters=20)
axes[0].contour(X1, X2, Z, levels=15, cmap='viridis', alpha=0.4)
axes[0].plot(traj_sgd[:, 0], traj_sgd[:, 1], 'ro-', linewidth=2, 
             markersize=6, label='SGD')
axes[0].scatter([0], [0], color='green', s=200, marker='*', zorder=5)
axes[0].set_title('标准SGD: 振荡严重')
axes[0].set_xlabel('x1')
axes[0].set_ylabel('x2')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axis('equal')

# Momentum轨迹
traj_momentum = momentum_2d(lr=0.2, beta=0.9, num_iters=20)
axes[1].contour(X1, X2, Z, levels=15, cmap='viridis', alpha=0.4)
axes[1].plot(traj_momentum[:, 0], traj_momentum[:, 1], 'bo-', 
             linewidth=2, markersize=6, label='Momentum (β=0.9)')
axes[1].scatter([0], [0], color='green', s=200, marker='*', zorder=5)
axes[1].set_title('动量法: 平滑路径')
axes[1].set_xlabel('x1')
axes[1].set_ylabel('x2')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].axis('equal')

plt.tight_layout()
plt.show()

print("\n动量法的优势:")
print("1. 减少了x2方向的振荡")
print("2. 加速了x1方向的进展")
print("3. 更快收敛到最优点")

## 2. AdaGrad - 自适应学习率

### 2.1 动机: 稀疏特征

**问题场景**: 自然语言处理、推荐系统
- 常见特征: "the", "is", "a" (频繁更新)
- 稀疏特征: "deep learning", "optimization" (罕见更新)

**统一学习率的问题**:
- 常见特征: 已经接近最优,不需要大步长
- 稀疏特征: 还需要探索,需要大步长

**解决方案**: 为每个参数维护独立的学习率

### 2.2 AdaGrad算法

**核心思想**: 根据历史梯度的大小自适应调整学习率

$$
\begin{aligned}
\mathbf{s}_t &= \mathbf{s}_{t-1} + \mathbf{g}_t^2 \\
\mathbf{w}_t &= \mathbf{w}_{t-1} - \frac{\eta}{\sqrt{\mathbf{s}_t + \epsilon}} \odot \mathbf{g}_t
\end{aligned}
$$

其中:
- $\mathbf{s}_t$: 累积梯度平方和
- $\epsilon$: 数值稳定项(通常$10^{-6}$)
- $\odot$: 逐元素乘法

**直观理解**:
- 梯度大的参数 → $\mathbf{s}_t$大 → 学习率小
- 梯度小的参数 → $\mathbf{s}_t$小 → 学习率大

### 2.3 AdaGrad的优缺点

**优点**:
- ✅ 自动调整学习率
- ✅ 适合稀疏数据
- ✅ 不同参数不同学习率

**缺点**:
- ❌ 学习率单调递减
- ❌ 后期更新太慢甚至停止
- ❌ 需要手动设置全局学习率$\eta$

In [ ]:
# AdaGrad实现与可视化
def adagrad_2d(lr, num_iters=20, x_init=(3.0, 3.0)):
    """AdaGrad算法"""
    x1, x2 = x_init
    s1, s2 = 0.0, 0.0
    eps = 1e-6
    trajectory = [(x1, x2)]
    
    for _ in range(num_iters):
        g1 = 0.2 * x1
        g2 = 4.0 * x2
        
        # 累积梯度平方
        s1 += g1**2
        s2 += g2**2
        
        # 自适应学习率更新
        x1 -= lr / math.sqrt(s1 + eps) * g1
        x2 -= lr / math.sqrt(s2 + eps) * g2
        trajectory.append((x1, x2))
    
    return np.array(trajectory)

# 对比不同算法
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

algorithms = [
    ('SGD', sgd_2d(lr=0.4)),
    ('Momentum', momentum_2d(lr=0.2, beta=0.9)),
    ('AdaGrad', adagrad_2d(lr=1.5))
]

for ax, (name, traj) in zip(axes, algorithms):
    ax.contour(X1, X2, Z, levels=15, cmap='viridis', alpha=0.4)
    ax.plot(traj[:, 0], traj[:, 1], 'o-', linewidth=2, markersize=6)
    ax.scatter([0], [0], color='red', s=200, marker='*', zorder=5)
    ax.set_title(f'{name}')
    ax.set_xlabel('x1')
    ax.set_ylabel('x2')
    ax.grid(True, alpha=0.3)
    ax.axis('equal')

plt.tight_layout()
plt.show()

print("AdaGrad特点:")
print("- 自动平衡不同方向的学习率")
print("- x2方向(梯度大)学习率快速衰减")
print("- x1方向(梯度小)保持较大学习率")

## 3. RMSProp - 解决AdaGrad的学习率衰减

### 3.1 AdaGrad的问题

AdaGrad中 $\mathbf{s}_t = \mathbf{s}_{t-1} + \mathbf{g}_t^2$ 单调递增:
- 学习率 $\frac{\eta}{\sqrt{\mathbf{s}_t}}$ 单调递减
- 后期几乎不更新
- 对非凸问题不友好

### 3.2 RMSProp算法

**核心改进**: 使用**指数移动平均**而非累积和

$$
\begin{aligned}
\mathbf{s}_t &= \gamma \mathbf{s}_{t-1} + (1-\gamma) \mathbf{g}_t^2 \\
\mathbf{w}_t &= \mathbf{w}_{t-1} - \frac{\eta}{\sqrt{\mathbf{s}_t + \epsilon}} \odot \mathbf{g}_t
\end{aligned}
$$

其中 $\gamma$ 通常取0.9或0.99。

**展开式**:
$$
\mathbf{s}_t = (1-\gamma)(\mathbf{g}_t^2 + \gamma \mathbf{g}_{t-1}^2 + \gamma^2 \mathbf{g}_{t-2}^2 + \cdots)
$$

**优势**:
- 学习率不再单调递减
- 更关注近期梯度
- 可以"忘记"早期的大梯度

### 3.3 AdaGrad vs RMSProp

In [ ]:
def rmsprop_2d(lr, gamma, num_iters=20, x_init=(3.0, 3.0)):
    """RMSProp算法"""
    x1, x2 = x_init
    s1, s2 = 0.0, 0.0
    eps = 1e-6
    trajectory = [(x1, x2)]
    
    for _ in range(num_iters):
        g1 = 0.2 * x1
        g2 = 4.0 * x2
        
        # 指数移动平均
        s1 = gamma * s1 + (1 - gamma) * g1**2
        s2 = gamma * s2 + (1 - gamma) * g2**2
        
        # 更新参数
        x1 -= lr / math.sqrt(s1 + eps) * g1
        x2 -= lr / math.sqrt(s2 + eps) * g2
        trajectory.append((x1, x2))
    
    return np.array(trajectory)

# 对比学习率变化
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# AdaGrad
traj_adagrad = adagrad_2d(lr=1.5, num_iters=30)
axes[0].contour(X1, X2, Z, levels=15, cmap='viridis', alpha=0.4)
axes[0].plot(traj_adagrad[:, 0], traj_adagrad[:, 1], 'o-', 
             linewidth=2, markersize=6, label='AdaGrad')
axes[0].scatter([0], [0], color='red', s=200, marker='*', zorder=5)
axes[0].set_title('AdaGrad: 后期几乎停止更新')
axes[0].set_xlabel('x1')
axes[0].set_ylabel('x2')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axis('equal')

# RMSProp
traj_rmsprop = rmsprop_2d(lr=0.4, gamma=0.9, num_iters=30)
axes[1].contour(X1, X2, Z, levels=15, cmap='viridis', alpha=0.4)
axes[1].plot(traj_rmsprop[:, 0], traj_rmsprop[:, 1], 'o-', 
             linewidth=2, markersize=6, label='RMSProp (γ=0.9)', color='orange')
axes[1].scatter([0], [0], color='red', s=200, marker='*', zorder=5)
axes[1].set_title('RMSProp: 持续更新到最优')
axes[1].set_xlabel('x1')
axes[1].set_ylabel('x2')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].axis('equal')

plt.tight_layout()
plt.show()

print("\nRMSProp改进:")
print("1. 学习率不会过早衰减")
print("2. 能够持续更新直到收敛")
print("3. 更适合非凸优化问题")

## 4. Adam - 集大成者

### 4.1 设计思想

**Adam** (Adaptive Moment Estimation) 结合了:
- **Momentum**: 一阶矩估计(梯度的指数移动平均)
- **RMSProp**: 二阶矩估计(梯度平方的指数移动平均)

### 4.2 算法详解

**状态变量**:
$$
\begin{aligned}
\mathbf{v}_t &= \beta_1 \mathbf{v}_{t-1} + (1-\beta_1) \mathbf{g}_t \\
\mathbf{s}_t &= \beta_2 \mathbf{s}_{t-1} + (1-\beta_2) \mathbf{g}_t^2
\end{aligned}
$$

**偏差修正**(Bias Correction):
$$
\begin{aligned}
\hat{\mathbf{v}}_t &= \frac{\mathbf{v}_t}{1 - \beta_1^t} \\
\hat{\mathbf{s}}_t &= \frac{\mathbf{s}_t}{1 - \beta_2^t}
\end{aligned}
$$

**参数更新**:
$$
\mathbf{w}_t = \mathbf{w}_{t-1} - \frac{\eta \hat{\mathbf{v}}_t}{\sqrt{\hat{\mathbf{s}}_t} + \epsilon}
$$

**超参数**:
- $\beta_1 = 0.9$ (一阶矩)
- $\beta_2 = 0.999$ (二阶矩)
- $\epsilon = 10^{-8}$ (数值稳定)
- $\eta = 0.001$ (学习率)

### 4.3 为什么需要偏差修正?

初始化: $\mathbf{v}_0 = 0, \mathbf{s}_0 = 0$

第一步:
$$
\mathbf{v}_1 = 0.9 \times 0 + 0.1 \mathbf{g}_1 = 0.1 \mathbf{g}_1
$$

**问题**: $\mathbf{v}_1$ 被低估了!

**修正**:
$$
\hat{\mathbf{v}}_1 = \frac{0.1 \mathbf{g}_1}{1 - 0.9^1} = \frac{0.1 \mathbf{g}_1}{0.1} = \mathbf{g}_1 \quad ✓
$$

In [ ]:
# Adam实现
def adam_2d(lr, beta1, beta2, num_iters=20, x_init=(3.0, 3.0)):
    """Adam算法"""
    x1, x2 = x_init
    v1, v2 = 0.0, 0.0
    s1, s2 = 0.0, 0.0
    eps = 1e-8
    trajectory = [(x1, x2)]
    
    for t in range(1, num_iters + 1):
        g1 = 0.2 * x1
        g2 = 4.0 * x2
        
        # 更新一阶矩和二阶矩
        v1 = beta1 * v1 + (1 - beta1) * g1
        v2 = beta1 * v2 + (1 - beta1) * g2
        s1 = beta2 * s1 + (1 - beta2) * g1**2
        s2 = beta2 * s2 + (1 - beta2) * g2**2
        
        # 偏差修正
        v1_hat = v1 / (1 - beta1**t)
        v2_hat = v2 / (1 - beta1**t)
        s1_hat = s1 / (1 - beta2**t)
        s2_hat = s2 / (1 - beta2**t)
        
        # 更新参数
        x1 -= lr * v1_hat / (math.sqrt(s1_hat) + eps)
        x2 -= lr * v2_hat / (math.sqrt(s2_hat) + eps)
        trajectory.append((x1, x2))
    
    return np.array(trajectory)

# 综合对比
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

optimizers = [
    ('SGD', sgd_2d(lr=0.4, num_iters=30)),
    ('Momentum', momentum_2d(lr=0.2, beta=0.9, num_iters=30)),
    ('AdaGrad', adagrad_2d(lr=1.5, num_iters=30)),
    ('RMSProp', rmsprop_2d(lr=0.4, gamma=0.9, num_iters=30)),
    ('Adam', adam_2d(lr=1.0, beta1=0.9, beta2=0.999, num_iters=30)),
]

# 绘制前5个子图
for i, (name, traj) in enumerate(optimizers):
    axes[i].contour(X1, X2, Z, levels=15, cmap='viridis', alpha=0.4)
    axes[i].plot(traj[:, 0], traj[:, 1], 'o-', linewidth=2, markersize=5)
    axes[i].scatter([0], [0], color='red', s=200, marker='*', zorder=5)
    axes[i].set_title(f'{name}', fontsize=14, fontweight='bold')
    axes[i].set_xlabel('x1')
    axes[i].set_ylabel('x2')
    axes[i].grid(True, alpha=0.3)
    axes[i].axis('equal')
    
    # 添加最终损失
    final_loss = 0.1 * traj[-1, 0]**2 + 2 * traj[-1, 1]**2
    axes[i].text(0.05, 0.95, f'Final Loss: {final_loss:.4f}', 
                 transform=axes[i].transAxes, fontsize=10,
                 verticalalignment='top', bbox=dict(boxstyle='round', 
                 facecolor='wheat', alpha=0.5))

# 第6个子图: 损失曲线对比
for name, traj in optimizers:
    losses = [0.1 * x1**2 + 2 * x2**2 for x1, x2 in traj]
    axes[5].plot(losses, linewidth=2, label=name)

axes[5].set_xlabel('Iteration')
axes[5].set_ylabel('Loss')
axes[5].set_title('收敛速度对比', fontsize=14, fontweight='bold')
axes[5].legend()
axes[5].grid(True, alpha=0.3)
axes[5].set_yscale('log')

plt.tight_layout()
plt.show()

print("\n优化器性能对比:")
print("\n算法特点:")
print("- SGD: 振荡严重,收敛慢")
print("- Momentum: 平滑路径,加速收敛")
print("- AdaGrad: 自适应但后期衰减过快")
print("- RMSProp: 解决AdaGrad衰减问题")
print("- Adam: 综合最优,收敛最快")
print("\n实践建议: Adam是大多数情况下的首选!")

## 5. Adadelta - 无学习率优化器

### 5.1 设计动机

**问题**: AdaGrad和RMSProp仍需手动设置全局学习率 $\eta$

**Adadelta的思想**: 用**参数变化量的尺度**自动确定学习率

### 5.2 算法

**状态变量**:
- $\mathbf{s}_t$: 梯度平方的指数移动平均
- $\Delta \mathbf{x}_t$: 参数变化量平方的指数移动平均

**更新规则**:
$$
\begin{aligned}
\mathbf{s}_t &= \rho \mathbf{s}_{t-1} + (1-\rho) \mathbf{g}_t^2 \\
\mathbf{g}_t' &= \frac{\sqrt{\Delta\mathbf{x}_{t-1} + \epsilon}}{\sqrt{\mathbf{s}_t + \epsilon}} \odot \mathbf{g}_t \\
\mathbf{x}_t &= \mathbf{x}_{t-1} - \mathbf{g}_t' \\
\Delta \mathbf{x}_t &= \rho \Delta\mathbf{x}_{t-1} + (1-\rho) (\mathbf{g}_t')^2
\end{aligned}
$$

**特点**:
- 没有全局学习率 $\eta$
- 学习率由参数变化历史决定
- 单位匹配更自然

### 5.3 实践应用

Adadelta在实践中较少使用,主要原因:
- 相比Adam没有明显优势
- 收敛速度通常不如Adam
- 调参空间小(只有$\rho$)但也意味着灵活性低

In [ ]:
# Adadelta实现
def adadelta_2d(rho, num_iters=30, x_init=(3.0, 3.0)):
    """Adadelta算法"""
    x1, x2 = x_init
    s1, s2 = 0.0, 0.0
    delta1, delta2 = 0.0, 0.0
    eps = 1e-6
    trajectory = [(x1, x2)]
    
    for _ in range(num_iters):
        g1 = 0.2 * x1
        g2 = 4.0 * x2
        
        # 更新梯度平方的移动平均
        s1 = rho * s1 + (1 - rho) * g1**2
        s2 = rho * s2 + (1 - rho) * g2**2
        
        # 计算调整后的梯度
        g1_prime = math.sqrt(delta1 + eps) / math.sqrt(s1 + eps) * g1
        g2_prime = math.sqrt(delta2 + eps) / math.sqrt(s2 + eps) * g2
        
        # 更新参数
        x1 -= g1_prime
        x2 -= g2_prime
        
        # 更新参数变化量的移动平均
        delta1 = rho * delta1 + (1 - rho) * g1_prime**2
        delta2 = rho * delta2 + (1 - rho) * g2_prime**2
        
        trajectory.append((x1, x2))
    
    return np.array(trajectory)

# 对比Adadelta和Adam
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Adadelta
traj_adadelta = adadelta_2d(rho=0.9)
axes[0].contour(X1, X2, Z, levels=15, cmap='viridis', alpha=0.4)
axes[0].plot(traj_adadelta[:, 0], traj_adadelta[:, 1], 'o-', 
             linewidth=2, markersize=6, label='Adadelta (ρ=0.9)')
axes[0].scatter([0], [0], color='red', s=200, marker='*', zorder=5)
axes[0].set_title('Adadelta: 无需设置学习率')
axes[0].set_xlabel('x1')
axes[0].set_ylabel('x2')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axis('equal')

# Adam (重新绘制)
traj_adam = adam_2d(lr=1.0, beta1=0.9, beta2=0.999)
axes[1].contour(X1, X2, Z, levels=15, cmap='viridis', alpha=0.4)
axes[1].plot(traj_adam[:, 0], traj_adam[:, 1], 'o-', 
             linewidth=2, markersize=6, label='Adam', color='purple')
axes[1].scatter([0], [0], color='red', s=200, marker='*', zorder=5)
axes[1].set_title('Adam: 当前最流行')
axes[1].set_xlabel('x1')
axes[1].set_ylabel('x2')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].axis('equal')

plt.tight_layout()
plt.show()

## 6. 小结

### 6.1 优化器演进

```
SGD
 |
 +-- Momentum ─────────┐
 |                     |
 +-- AdaGrad ───┬──── Adam (最优)
                |      |
                +──── RMSProp
                |
                +──── Adadelta
```

### 6.2 核心思想对比

| 优化器 | 核心机制 | 关键超参数 | 适用场景 |
|--------|----------|------------|----------|
| SGD | 负梯度方向 | $\eta$ | 凸优化,需精细调参 |
| Momentum | 累积历史梯度 | $\eta, \beta$ | 加速收敛,平滑路径 |
| AdaGrad | 自适应学习率 | $\eta$ | 稀疏数据,早期训练 |
| RMSProp | 梯度平方指数移动平均 | $\eta, \gamma$ | 非凸问题,RNN |
| Adam | Momentum + RMSProp | $\eta, \beta_1, \beta_2$ | 通用,大多数任务 |
| Adadelta | 无学习率 | $\rho$ | 理论研究 |

### 6.3 状态变量对比

| 优化器 | 状态变量 | 内存开销 |
|--------|----------|----------|
| SGD | 无 | 0 |
| Momentum | $\mathbf{v}$ | $O(d)$ |
| AdaGrad | $\mathbf{s}$ | $O(d)$ |
| RMSProp | $\mathbf{s}$ | $O(d)$ |
| Adam | $\mathbf{v}, \mathbf{s}$ | $O(2d)$ |
| Adadelta | $\mathbf{s}, \Delta\mathbf{x}$ | $O(2d)$ |

### 6.4 实践建议

**首选**: Adam ($\eta=10^{-3}, \beta_1=0.9, \beta_2=0.999$)
- 鲁棒性好
- 收敛快
- 适用广泛

**备选**:
- **SGD + Momentum**: 最终性能可能更好(需精细调参)
- **RMSProp**: RNN任务
- **AdaGrad**: 稀疏数据(如NLP)

**调参顺序**:
1. 学习率 $\eta$ (最重要!)
2. 批量大小
3. 动量系数 $\beta$
4. 其他超参数

### 6.5 深入理解

**动量方法**的本质:
- 平滑梯度噪声
- 加速一致方向
- 减缓振荡方向

**自适应学习率**的本质:
- 为不同参数设置不同学习率
- 梯度大 → 学习率小
- 梯度小 → 学习率大

**Adam成功的原因**:
- 结合了动量和自适应的优点
- 偏差修正解决了初始化问题
- 默认超参数就很好

### 下一步

- **学习率调度**: 预热、余弦退火、分段衰减
- **优化器变体**: AdamW, Lookahead, RAdam
- **实践技巧**: 梯度裁剪、权重衰减

## 练习

1. **动量系数实验**: 在病态问题上测试不同$\beta$值(0.5, 0.9, 0.99)的效果。

2. **AdaGrad衰减**: 绘制AdaGrad的学习率随迭代变化曲线,观察衰减速度。

3. **偏差修正**: 实现有/无偏差修正的Adam,对比前10步的更新量差异。

4. **优化器对比**: 在真实数据集(如MNIST)上比较所有优化器的性能。

5. **稀疏梯度**: 构造90%参数梯度为0的场景,测试AdaGrad vs Adam。